# 01 - Data Cleaning: Hotel Booking Cancellations

In this notebook we load the raw file *hotels.csv*, fix the data types, handle the missing values, remove invalid rows and save a clean file, *hotels_clean.csv*, for the modeling step.

In [1]:
import pandas as pd

In [2]:
csv_path = "../data/hotels.csv"
df = pd.read_csv(csv_path)

In [3]:
df.shape

(119390, 32)

In [4]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 1. Data types
We check the type of each column and fix the ones that do not match what the data means.

In [5]:
df.dtypes.to_frame("dtype")


,dtype
hotel,str
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,str
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64


In [6]:
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])


In [7]:
df["children"] = df["children"].astype("Int64")

In [8]:
for col in ["agent", "company"]:
    df[col] = df[col].astype("Int64").astype("string")

In [9]:
df.dtypes.to_frame("dtype")


,dtype
hotel,str
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,str
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64


**Conclusion**. Four columns had the wrong type.
- *Reservation_status_date*: was text, we changed it to a date.

- *Children*: was decimal, as it is impossible to have a number of children decimal, we changed it to an integer.

- *Agent* and *Company*: are ID codes, not quantities. Because *Agent* 240 is not more than *Agent* 120, so we changed them to text.

## 2. Missing values
We count the missing values in each column and decide how to fill them.

In [10]:
missing = df.isna().sum()
missing[missing > 0]

children         4
country        488
agent        16340
company     112593
dtype: int64

In [11]:
print(df["children"].value_counts())

children
0     110796
1       4861
2       3652
3         76
10         1
Name: count, dtype: Int64


In [12]:
for col in ["country", "agent", "company"]:
    print(col, "->", df[col].nunique(), "distinct values")
    print(df[col].value_counts().head())
    print()

country -> 177 distinct values
country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
Name: count, dtype: int64

agent -> 333 distinct values
agent
9      31961
240    13922
1       7191
14      3640
7       3539
Name: count, dtype: int64[pyarrow]

company -> 352 distinct values
company
40     927
223    784
67     267
45     250
153    215
Name: count, dtype: int64[pyarrow]



In [13]:
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")
df["agent"] = df["agent"].fillna("No agent")
df["company"] = df["company"].fillna("No company")

**Conclusion**. Four columns have missing values: *children* (4 rows), *country* (488), *agent* (16,340) and *company* (112,593).

**Why we filled them and didn't remove rows**
- In *agent* and *company*, missing means the booking had no agent or no company. It is not an error and removing those rows would lose 94% of the data for *company* and 14% for *agent*.

- *Children* is 0 in about 93% of the bookings, so 0 is the most reasonable value.

- *Country* is filled with "Unknown" because we cannot guess it.

## 3. Duplicates
We look for rows that are exactly the same in all columns and decide if we keep them.

In [14]:
n_duplicates = df.duplicated().sum()
print(n_duplicates)

31994


In [15]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(6)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40772,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14,No company,0,Transient,75.0,0,1,Check-Out,2015-08-09
40802,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14,No company,0,Transient,75.0,0,1,Check-Out,2015-08-09
40821,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9,No company,0,Transient,89.0,0,1,Check-Out,2015-08-09
40838,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9,No company,0,Transient,89.0,0,1,Check-Out,2015-08-09
76792,City Hotel,0,0,2015,August,33,10,1,0,2,...,No Deposit,No agent,No company,0,Transient,75.0,0,0,Check-Out,2015-08-11
76793,City Hotel,0,0,2015,August,33,10,1,0,2,...,No Deposit,No agent,No company,0,Transient,75.0,0,0,Check-Out,2015-08-11


In [16]:
dup = df.duplicated()
print("duplicates:", round(dup.mean() * 100, 1), "%")
print("cancel rate in duplicates:", round(df[dup]["is_canceled"].mean() * 100, 1), "%")
print("cancel rate before:", round(df["is_canceled"].mean() * 100, 1), "%")
print("cancel rate after:", round(df[~dup]["is_canceled"].mean() * 100, 1), "%")
print(dup.groupby(df["market_segment"]).mean().round(2))

duplicates: 26.8 %
cancel rate in duplicates: 63.1 %
cancel rate before: 37.0 %
cancel rate after: 27.5 %
market_segment
Aviation         0.04
Complementary    0.06
Corporate        0.20
Direct           0.06
Groups           0.75
Offline TA/TO    0.43
Online TA        0.09
Undefined        0.00
dtype: float64


In [17]:
df = df.drop_duplicates().reset_index(drop=True)

**Conclusion**
- 31,994 rows (26.8 %) are exact duplicates across all 32 columns.
- There is no booking ID, so we cannot prove whether they are repeated records or real bookings.
- Duplicates are concentrated in the *Groups* segment (75 % of its rows) and are canceled more often (63 % vs 27.5 %). Some may be real multi-room group bookings.

**Why we removed them**
- Keeping them could repeat the same row in the training and test sets and inflate model performance.
- The dataset goes from 119,390 to 87,396 rows.
- Trade-off: removing them lowers the cancellation rate from 37.0 % to 27.5 %, so the clean data has fewer cancellations than the raw data.

## 4. Invalid values

We count the values that do not make sense. Prices are studied here and the other problems in sections 5 to 7.

In [18]:
total_nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
total_guests = df["adults"] + df["children"] + df["babies"]


print("adr < 0:", (df["adr"] < 0).sum())
print("adr >= 1000:", (df["adr"] >= 1000).sum())
print("adr == 0:", (df["adr"] == 0).sum())
print("zero nights:", (total_nights == 0).sum())
print("zero guests:", (total_guests == 0).sum())


print(df[["adults", "children", "babies"]].max())

adr < 0: 1
adr >= 1000: 1
adr == 0: 1778
zero nights: 651
zero guests: 166
adults      55
children    10
babies      10
dtype: Int64


In [19]:
df[df["adr"] < 0].T

,11591
hotel,Resort Hotel
is_canceled,0
lead_time,195
arrival_date_year,2017
arrival_date_month,March
arrival_date_week_number,10
arrival_date_day_of_month,5
stays_in_weekend_nights,4
stays_in_week_nights,6
adults,2


In [20]:
df = df[df["adr"] >= 0].reset_index(drop=True)
print(df.shape)

(87395, 32)


### 4a. Negative price

**Conclusion.** *adr* is the average daily rate, the price per night (300 € for 3 nights gives an *adr* of 100). One booking has an *adr* of -6.38.

**Why we removed it.** A negative price is not possible, so it is an error.

In [21]:
print(df["adr"].nlargest(5))

38748    5400.0
80727     510.0
11660     508.0
74460     451.5
9984      450.0
Name: adr, dtype: float64


In [22]:
df[df["adr"] >= 1000].T

,38748
hotel,City Hotel
is_canceled,1
lead_time,35
arrival_date_year,2016
arrival_date_month,March
arrival_date_week_number,13
arrival_date_day_of_month,25
stays_in_weekend_nights,0
stays_in_week_nights,1
adults,2


In [23]:
df = df[df["adr"] < 1000].reset_index(drop=True)
print(df.shape)

(87394, 32)


### 4b. Very high price

**Conclusion.** One booking has an *adr* of 5,400, a price per night far above the rest of the data. The next higher *adr* is 510.

**Why we removed it.** A price this high is very likely an error or an outlier, and a single extreme value can distort what the model learns. We chose the limit (1,000) by hand.

## 5. Zero nights

We study the bookings where weekend nights + week nights = 0.

In [24]:
total_nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

zero_nights = df[total_nights == 0]
print(len(zero_nights))


print(zero_nights["reservation_status"].value_counts())
print(zero_nights["market_segment"].value_counts())
print(zero_nights["adr"].describe())

651
reservation_status
Check-Out    622
Canceled      16
No-Show       13
Name: count, dtype: int64
market_segment
Online TA        297
Direct           131
Offline TA/TO    122
Corporate         47
Groups            34
Complementary     15
Aviation           5
Name: count, dtype: int64
count    651.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: adr, dtype: float64


In [25]:
df = df[total_nights != 0].reset_index(drop=True)
print(df.shape)

(86743, 32)


**Conclusion.** 651 bookings have 0 nights, and all of them have an *adr* of 0, so no night was charged.

**Why we removed them.** Most (622) are marked Check-Out, but with 0 nights and an *adr* of 0 they carry no revenue and no stay length, so they are probably recording errors.

## 6. Zero guests

We study the bookings where adults + children + babies = 0.

In [26]:
total_guests = df["adults"] + df["children"] + df["babies"]


zero_guests = df[total_guests == 0]
print(len(zero_guests))

print(zero_guests["reservation_status"].value_counts())
print(zero_guests["market_segment"].value_counts())
print((zero_guests["adr"] == 0).sum())

106
reservation_status
Check-Out    96
Canceled      9
No-Show       1
Name: count, dtype: int64
market_segment
Online TA        36
Direct           21
Offline TA/TO    18
Groups           16
Corporate        10
Complementary     5
Name: count, dtype: int64
75


In [27]:
df = df[total_guests != 0].reset_index(drop=True)
print(df.shape)

(86637, 32)


**Conclusion.** 106 bookings have 0 guests, and 75 of them also have an *adr* of 0. The overview showed 166, but 60 of them also had 0 nights and were already removed in section 5.

**Why we removed them.** A booking without any guest is not valid.

## 7. Extreme number of guests

We look for bookings with a very large number of adults, children or babies.

In [28]:
for col in ["adults", "children", "babies"]:
    print(df[col].value_counts().sort_index(), "\n")

adults
0       219
1     16317
2     64106
3      5919
4        60
5         2
6         1
10        1
20        2
26        5
27        2
40        1
50        1
55        1
Name: count, dtype: int64 

children
0     78306
1      4676
2      3579
3        75
10        1
Name: count, dtype: Int64 

babies
0     85728
1       892
2        15
9         1
10        1
Name: count, dtype: int64 



In [29]:

extreme = df[(df["adults"] >= 10) | (df["children"] >= 9) | (df["babies"] >= 9)]
print(len(extreme))

extreme[["hotel", "adults", "children", "babies", "market_segment", "adr", "is_canceled"]]

16


,hotel,adults,children,babies,market_segment,adr,is_canceled
314,Resort Hotel,2,10,0,Offline TA/TO,133.16,1
1426,Resort Hotel,40,0,0,Direct,0.00,1
1474,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
1528,Resort Hotel,50,0,0,Direct,0.00,1
1634,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
1757,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
1785,Resort Hotel,27,0,0,Direct,0.00,1
1828,Resort Hotel,27,0,0,Direct,0.00,1
1866,Resort Hotel,26,0,0,Offline TA/TO,0.00,1
2013,Resort Hotel,26,0,0,Offline TA/TO,0.00,1


In [30]:
df = df.drop(extreme.index).reset_index(drop=True)
print(df.shape)

(86621, 32)


**Conclusion.** 16 bookings have 10 or more adults, or 9 or more children or babies. For example, one booking has 10 children while the next highest value is 3. For *adults* and *babies*, the next highest values are 6 and 2. Of these 16, 14 were canceled and 13 have an *adr* of 0.

**Why we removed them.** These values are very far from the rest and are probably errors or test bookings. We chose the limits (10 and 9) by hand.

## 8. Zero adults

We study the bookings with 0 adults, which are not removed by the zero guests check because they have children.

In [31]:

zero_adults = df[df["adults"] == 0]
print(len(zero_adults))


print(zero_adults["hotel"].value_counts())
print(zero_adults["children"].value_counts())
print(zero_adults["reserved_room_type"].value_counts().head())
print(zero_adults["adr"].median())

219
hotel
City Hotel    219
Name: count, dtype: int64
children
2    204
3     11
1      4
Name: count, dtype: Int64
reserved_room_type
B    205
A      9
F      2
G      1
E      1
Name: count, dtype: int64
87.08


**Conclusion.** 219 bookings have 0 adults but do have children (204 of them have 2 children). This is 0.25 % of the data. All are in City Hotel and the median *adr* is 87.08, a normal price.

**Why we kept them.** They have guests and a normal price, so they are not clearly errors.

## 9. Zero price

We check how many bookings still have an *adr* of 0 after the previous steps and which segments they belong to.

In [32]:
print((df["adr"] == 0).sum())
print(df[df["adr"] == 0]["market_segment"].value_counts())

1039
market_segment
Complementary    620
Groups           171
Offline TA/TO    117
Direct            72
Online TA         40
Corporate         19
Name: count, dtype: int64


**Conclusion.** After the previous steps, 1,039 bookings (1.2 % of the data) still have an *adr* of 0. 620 of them are in the *Complementary* segment, which are free stays. The other 419 may be groups without a price or errors, but we cannot separate them. The overview showed 1,778, but 739 were already removed in sections 5 to 7 (651 with 0 nights, 75 with 0 guests and 13 with extreme guest counts).

**Why we kept them.** A free stay is a real booking, and the rest are only a small part of the data. We leave the *adr* as 0 because we cannot know their real price, and filling it with the mean or median would invent a value. The modeling step can test other options.

## 10. Target leakage

We compare *reservation_status* with *is_canceled* to check if one column gives away the other.

In [33]:
pd.crosstab(df["reservation_status"], df["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,22972
Check-Out,62650,0
No-Show,0,999


**Conclusion.** *reservation_status* matches *is_canceled* exactly. Canceled and No-Show are always 1, and Check-Out is always 0.

**Why we exclude it.** The column contains the answer we want to predict, so a model would copy it and look perfect. It is only known after the booking ends. *reservation_status_date*, the date of that final status, is excluded for the same reason.

**What we do with them.** We keep both columns in the file and leave them out when choosing the model features, which is done in the modeling step.

## 11. Final checks and saving

We check that the dataset has no missing values, no duplicates and no invalid prices. Then we save it as *hotels_clean.csv* and load it again to confirm that the shape is correct.

In [34]:
print(df.shape)
print(df.isna().sum().sum())
print(df.duplicated().sum())
print((df["adr"] < 0).sum())
print((df["adr"] >= 1000).sum())

(86621, 32)
0
0
0
0


In [35]:
clean_path = "../data/hotels_clean.csv"
df.to_csv(clean_path, index=False)

In [36]:
check = pd.read_csv(clean_path)
print(check.shape)

(86621, 32)


**Conclusion.** The saved file has 86,621 rows and 32 columns, the same as the dataframe, with no missing values and no duplicates.

## 12. Cleaning summary

| Step | Rows removed | Rows left |
|---|---|---|
| Start | 0 | 119,390 |
| Duplicates | 31,994 | 87,396 |
| Negative *adr* | 1 | 87,395 |
| *adr* >= 1000 | 1 | 87,394 |
| Zero nights | 651 | 86,743 |
| Zero guests | 106 | 86,637 |
| Extreme number of guests | 16 | 86,621 |

In total we removed 32,769 rows (27.4 %).